# 01 — Data Exploration

Milestone 1 (session wrap-up): load historical adjusted close prices, inspect coverage, and compute simple/log returns.

**Note:** Yahoo Finance downloads can time out. This notebook prefers the local cache at `data/cache/prices_close.csv`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Allow importing from ../src when the notebook runs from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import (
    DEFAULT_BENCHMARK,
    DEFAULT_START,
    DEFAULT_TICKERS,
    load_or_download,
)

TICKERS = DEFAULT_TICKERS
BENCHMARK = DEFAULT_BENCHMARK
ALL_TICKERS = TICKERS + [BENCHMARK]
START = DEFAULT_START

print("Project root:", PROJECT_ROOT)
print("Universe:", ALL_TICKERS)

## Load prices

Uses the cached CSV by default. Set `refresh=True` only when you want to re-download from Yahoo Finance.

In [ ]:
prices = load_or_download(
    tickers=ALL_TICKERS,
    start=START,
    refresh=False,  # set True to force a fresh Yahoo download
)

print("shape:", prices.shape)
print("date range:", prices.index.min().date(), "→", prices.index.max().date())
print("missing values per ticker:")
print(prices.isna().sum())
prices.head()

## Quick price chart

Normalized to 1.0 at the first date so different price levels are comparable.

In [ ]:
normalized = prices / prices.iloc[0]

ax = normalized.plot(figsize=(12, 6), title="Normalized adjusted close (start = 1.0)")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

## Returns

- **Simple return:** $r_t = P_t / P_{t-1} - 1$ (portfolio math uses these)
- **Log return:** $\log(P_t / P_{t-1})$ (handy for compounding / stats)

The first row is NaN because there is no previous price.

In [ ]:
simple_returns = prices.pct_change()
log_returns = np.log(prices / prices.shift(1))

summary = pd.DataFrame({
    "mean_daily_simple": simple_returns.mean(),
    "std_daily_simple": simple_returns.std(),
    "ann_return_approx": simple_returns.mean() * 252,
    "ann_vol_approx": simple_returns.std() * np.sqrt(252),
})

summary.round(4)

## Correlation of daily simple returns

Diversification works better when assets do not all move together.

In [ ]:
corr = simple_returns.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Daily return correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

corr.round(2)

## Session checkpoint

Done today:
- Project setup + virtualenv
- Price universe defined (9 stocks + SPY)
- Cached adjusted closes loaded
- Simple/log returns + basic summary + correlation

Next session:
- Equal-weight baseline portfolio vs SPY
- Risk metrics (Sharpe, max drawdown)
- Minimum-variance / max-Sharpe optimizers